In [11]:
!pip install transformers tqdm pandas openai
from openai import OpenAI
from transformers import AutoTokenizer
import pandas as pd
from tqdm import tqdm
import json
import requests


[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: pip install --upgrade pip


## API

In [12]:
# client = OpenAI(
#     api_key="sk-452b56e0803f461f80e83eaeeb1b7146",
#     base_url="https://dashscope-intl.aliyuncs.com/compatible-mode/v1",
# )

def call_llm(prompt):
    url = "http://127.0.0.1:11434/api/generate"

    payload = {
        "model": "qwen2.5:7b-instruct",   # 🚨 注意：冒号，不是横杠
        "prompt": prompt,
        "stream": False,
        "format": "json"                  # 🚨 强制 JSON 输出
    }

    r = requests.post(url, json=payload, timeout=60)
    r.raise_for_status()

    data = r.json()
    return data["response"]   # 返回 JSON 字符串

# 使用 BERT tokenizer（最稳定、可用）
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

# def count_tokens(text):
#     tokens = tokenizer.encode(
#         text,
#         add_special_tokens=False,
#         truncation=False,   # 🔥 不截断
#         max_length=99999    # 🔥 防止 512 限制触发
#     )
#     return len(tokens)

In [13]:
df = pd.read_csv("sporc_turns_selected_clean.csv")
df = df[['episode', 'turn', 'speaker', 'role', 'text']]
# 按 episode + turn 排序
df = df.sort_values(by=['episode','turn'])

# 计算所有 episode_id
episode_ids = df['episode'].unique()

# 计算 turn 分布（每集最大的 turn_id = turn 数）
turn_counts = df.groupby('episode')['turn'].max()

# 打印总共有多少集
print(f"\n📌 数据集中一共有 {len(episode_ids)} 集")

# turn 分布统计
print("📌 turn 数量统计分布：")
print(f" - 最少 turn 数: {turn_counts.min()}")
print(f" - 最多 turn 数: {turn_counts.max()}")
print(f" - 平均 turn 数: {turn_counts.mean():.2f}")
print(f" - 中位数 turn 数: {turn_counts.median():.2f}")
print(f" - 90% 分位 turn 数: {turn_counts.quantile(0.9):.2f}")
print(f" - 95% 分位 turn 数: {turn_counts.quantile(0.95):.2f}")
print(f" - 99% 分位 turn 数: {turn_counts.quantile(0.99):.2f}\n")

episodes = df.groupby("episode")


# role 为空的判断要同时排除 NaN 和 空字符串 ""
missing_role_df = df[(df['role'].isna()) | (df['role'].astype(str).str.strip() == "")]

missing_count = len(missing_role_df)

print(f"📌 role 缺失的文本数（role 为空或 NaN）：{missing_count}")

# 如果你想看占比（可选）
print(f"📌 占全部记录比例：{missing_count / len(df) * 100:.4f}%")

# 如果你想看这些 episode（可选）
missing_episode_ids = missing_role_df['episode'].unique()
print(f"📌 出现 role 缺失的 episode 数：{len(missing_episode_ids)} 个")

# 先把 role 全部转成字符串，strip 掉空白
df['role_clean'] = df['role'].astype(str).str.strip().str.lower()

# 统计所有 unique role
role_counts = df['role_clean'].value_counts(dropna=False)

print("\n📌 全部出现过的 role 值及数量：")
for role, cnt in role_counts.items():
    print(f" - '{role}': {cnt}")

# =============================================================================
# 📌 检查是否存在异常 role（不是 host / guest）
# =============================================================================
valid_roles = {"host", "guest"}

abnormal_roles = [role for role in role_counts.index if role not in valid_roles]

print("\n📌 异常 role 值（不是 host/guest 的）：")
if len(abnormal_roles) == 0:
    print(" - 👍 没有发现异常 role，数据干净。")
else:
    for r in abnormal_roles:
        print(f" - '{r}' 出现 {role_counts[r]} 次")


📌 数据集中一共有 138050 集
📌 turn 数量统计分布：
 - 最少 turn 数: 1
 - 最多 turn 数: 5940
 - 平均 turn 数: 185.86
 - 中位数 turn 数: 77.00
 - 90% 分位 turn 数: 515.00
 - 95% 分位 turn 数: 762.00
 - 99% 分位 turn 数: 1365.51

📌 role 缺失的文本数（role 为空或 NaN）：0
📌 占全部记录比例：0.0000%
📌 出现 role 缺失的 episode 数：0 个

📌 全部出现过的 role 值及数量：
 - 'host': 5810050
 - 'guest': 1452828

📌 异常 role 值（不是 host/guest 的）：
 - 👍 没有发现异常 role，数据干净。


In [14]:

# 计算 turn 分布（每集最大的 turn_id = turn 数）
turn_counts = df.groupby('episode')['turn'].max()

# 打印总共有多少集
print(f"\n📌 数据集中一共有 {len(episode_ids)} 集")

# turn 分布统计
print("📌 turn 数量统计分布：")
print(f" - 最少 turn 数: {turn_counts.min()}")
print(f" - 最多 turn 数: {turn_counts.max()}")
print(f" - 平均 turn 数: {turn_counts.mean():.2f}")
print(f" - 中位数 turn 数: {turn_counts.median():.2f}")
print(f" - 90% 分位 turn 数: {turn_counts.quantile(0.9):.2f}")
print(f" - 95% 分位 turn 数: {turn_counts.quantile(0.95):.2f}")
print(f" - 99% 分位 turn 数: {turn_counts.quantile(0.99):.2f}\n")

episodes = df.groupby("episode")


# role 为空的判断要同时排除 NaN 和 空字符串 ""
missing_role_df = df[(df['role'].isna()) | (df['role'].astype(str).str.strip() == "")]

missing_count = len(missing_role_df)

print(f"📌 role 缺失的文本数（role 为空或 NaN）：{missing_count}")

# 如果你想看占比（可选）
print(f"📌 占全部记录比例：{missing_count / len(df) * 100:.4f}%")

# 如果你想看这些 episode（可选）
missing_episode_ids = missing_role_df['episode'].unique()
print(f"📌 出现 role 缺失的 episode 数：{len(missing_episode_ids)} 个")

# 先把 role 全部转成字符串，strip 掉空白
df['role_clean'] = df['role'].astype(str).str.strip().str.lower()

# 统计所有 unique role
role_counts = df['role_clean'].value_counts(dropna=False)

print("\n📌 全部出现过的 role 值及数量：")
for role, cnt in role_counts.items():
    print(f" - '{role}': {cnt}")

# =============================================================================
# 📌 检查是否存在异常 role（不是 host / guest）
# =============================================================================
valid_roles = {"host", "guest"}

abnormal_roles = [role for role in role_counts.index if role not in valid_roles]

print("\n📌 异常 role 值（不是 host/guest 的）：")
if len(abnormal_roles) == 0:
    print(" - 👍 没有发现异常 role，数据干净。")
else:
    for r in abnormal_roles:
        print(f" - '{r}' 出现 {role_counts[r]} 次")


📌 数据集中一共有 138050 集
📌 turn 数量统计分布：
 - 最少 turn 数: 1
 - 最多 turn 数: 5940
 - 平均 turn 数: 185.86
 - 中位数 turn 数: 77.00
 - 90% 分位 turn 数: 515.00
 - 95% 分位 turn 数: 762.00
 - 99% 分位 turn 数: 1365.51

📌 role 缺失的文本数（role 为空或 NaN）：0
📌 占全部记录比例：0.0000%
📌 出现 role 缺失的 episode 数：0 个

📌 全部出现过的 role 值及数量：
 - 'host': 5810050
 - 'guest': 1452828

📌 异常 role 值（不是 host/guest 的）：
 - 👍 没有发现异常 role，数据干净。


## Token-based chunking

## Prompt

In [17]:
def call_llm_single_episode(text):
    url = "http://127.0.0.1:11434/api/generate"

    prompt = f"""
You are a podcast summarization model.

You will read the entire episode transcript.
Each line starts with a role tag: host or guest.

Return STRICT JSON, no commentary:

{{
  "global_summary": "...",
  "host_summary": "...",
  "guest_summary": "..."
}}

Transcript:
{text}
"""

    payload = {
        "model": "qwen2.5:7b-instruct",
        "prompt": prompt,
        "format": "json",
        "stream": False
    }

    r = requests.post(url, json=payload, timeout=120)
    r.raise_for_status()
    return json.loads(r.json()["response"])


In [ ]:
def summarize_episode(ep_id, df_ep):
    # 拼接整集文本（带 role）
    lines = [
        f"{role}: {text}"
        for role, text in zip(df_ep["role"].tolist(), df_ep["text"].tolist())
    ]
    full_text = "\n".join(lines)

    try:
        result = call_llm_single_episode(full_text)
        result["episode"] = ep_id
        return result
    except Exception as e:
        return {
            "episode": ep_id,
            "global_summary": "",
            "host_summary": "",
            "guest_summary": ""
        }

In [19]:
import os
import time

OUTPUT_FILE = "sporc_full_summaries_7b.jsonl"
SAVE_EVERY = 50
MAX_RETRY = 8 

# ----- 断点续跑 -----
existing_episodes = set()
if os.path.exists(OUTPUT_FILE):
    with open(OUTPUT_FILE, "r") as f:
        for line in f:
            try:
                existing_episodes.add(json.loads(line)["episode"])
            except:
                pass

print(f"📌 已处理 episode 数量: {len(existing_episodes)}")

buffer = []

# ----- 主循环 -----
for ep_id in tqdm(episode_ids, desc="Processing Episodes"):
    if ep_id in existing_episodes:
        continue

    df_ep = episodes.get_group(ep_id)

    for attempt in range(MAX_RETRY):
        try:
            result = summarize_episode(ep_id, df_ep)
            buffer.append(result)
            break

        except json.JSONDecodeError:
            print(f"⚠️ Episode {ep_id}: LLM JSON 格式错误，重试({attempt+1})")
            time.sleep(1)

        except Exception as e:
            print(f"⚠️ Episode {ep_id}: 其他错误({attempt+1}) → {e}")
            time.sleep(1)

    else:
        # 所有 retry 都失败
        buffer.append({
            "episode": ep_id,
            "global_summary": "",
            "host_summary": "",
            "guest_summary": ""
        })

    # ----- 批量保存 -----
    if len(buffer) >= SAVE_EVERY:
        with open(OUTPUT_FILE, "a") as f:
            for r in buffer:
                f.write(json.dumps(r, ensure_ascii=False) + "\n")
        buffer = []

# ----- 最后 flush -----
if buffer:
    with open(OUTPUT_FILE, "a") as f:
        for r in buffer:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")

print("🎉 全量任务完成！")


📌 已处理 episode 数量: 250


Processing Episodes:   7%|▋         | 9376/138050 [00:06<01:14, 1719.32it/s]Exception ignored in: <finalize object at 0x1f87f70e0; dead>
Traceback (most recent call last):
  File "/opt/anaconda3/lib/python3.12/weakref.py", line 585, in __call__
    def __call__(self, _=None):

KeyboardInterrupt: 
Processing Episodes:  44%|████▎     | 60097/138050 [00:34<00:45, 1727.26it/s]


KeyboardInterrupt: 

In [104]:
from multiprocessing import Pool, cpu_count

NUM_WORKERS = 5   # M4 Pro 推荐 4–6

def process_one(ep_id):
    if ep_id in existing_episodes:
        return None

    df_ep = episodes.get_group(ep_id)
    return summarize_episode(ep_id, df_ep)


In [105]:
SAVE_EVERY = 1
OUTPUT_FILE = "sporc_fast_summaries.jsonl"

existing_episodes = set()
if os.path.exists(OUTPUT_FILE):
    with open(OUTPUT_FILE, "r") as f:
        for line in f:
            try:
                existing_episodes.add(json.loads(line)["episode"])
            except:
                pass

print("已处理 episode 数量:", len(existing_episodes))

with Pool(NUM_WORKERS) as p:
    for result in p.imap_unordered(process_one, episode_ids):
        if result is None:
            continue
        with open(OUTPUT_FILE, "a") as f:
            f.write(json.dumps(result, ensure_ascii=False) + "\n")


已处理 episode 数量: 0


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment

KeyboardInterrupt: 

In [2]:
import json, os, pandas as pd, requests

OUTPUT_FILE = "sporc_full_summaries_7b.jsonl"   # ✔ 用你的文件名

def call_llm(text):
    url = "http://127.0.0.1:11434/api/generate"
    prompt = f"""
You are a podcast summarizer.

Return JSON:
{{
 "global_summary": "...",
 "host_summary": "...",
 "guest_summary": ""
}}

Transcript:
{text}
"""
    payload = {
        "model": "qwen2.5:7b-instruct",
        "prompt": prompt,
        "format": "json",
        "stream": False
    }
    r = requests.post(url, json=payload, timeout=120)
    r.raise_for_status()
    return json.loads(r.json()["response"])

def summarize_episode(ep_id, df_ep):
    lines = [f"{r}: {t}" for r,t in zip(df_ep["role"], df_ep["text"])]
    full = "\n".join(lines)
    try:
        result = call_llm(full)
        result["episode"] = ep_id
        return result
    except:
        return {"episode": ep_id, "global_summary":"", "host_summary":"","guest_summary":""}

def main():
    # 👇 正确的 CSV 文件名
    df = pd.read_csv("sporc_turns_selected_clean.csv")
    episodes = df.groupby("episode")

    # 断点续跑：加载你之前跑好的 250 集
    existing = set()
    if os.path.exists(OUTPUT_FILE):
        for line in open(OUTPUT_FILE):
            try:
                existing.add(json.loads(line)["episode"])
            except:
                pass

    print("已处理:", len(existing))

    # 单进程先跑通
    with open(OUTPUT_FILE, "a") as f:
        for ep_id, df_ep in episodes:
            if ep_id in existing:
                continue
            result = summarize_episode(ep_id, df_ep)
            f.write(json.dumps(result, ensure_ascii=False) + "\n")

if __name__ == "__main__":
    main()


KeyboardInterrupt: 